# T07 (revised) — Backtracking input adapter / particle interface, manual check

**Correction notice:** the original version of this notebook demonstrated a synthetic
observation footprint and particle *generation*. That has been removed: satellite
segmentation and particle generation belong to an upstream teammate module, not to this
backtracking codebase. This notebook now starts from a small **mock upstream particle
payload** (standing in for that teammate's output) and shows only the adapter that
validates and hands it off into our existing T05/T06 structures.

This notebook does **not** demonstrate satellite segmentation, oil/clean
classification, or particle creation of any kind.

In [15]:
import os
import sys

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

import numpy as np

from iste.environment import GriddedFieldSource
from iste.observation import REQUIRED_UPSTREAM_FIELDS, adapt_upstream_particles, tag_trajectory
from iste.trajectory import run_backward_trajectory

LAT = np.array([10.0, 20.0, 30.0])
LON = np.array([100.0, 110.0, 120.0])
T0 = np.datetime64("2020-01-01T06:00", "ns")
TIME = T0 - np.arange(5)[::-1] * np.timedelta64(3600, "s")
DT = 900.0
ZERO_K = np.zeros((3, 3))


def make_ocean(uo_grid, vo_grid):
    n_t = len(TIME)
    uo = np.stack([uo_grid] * n_t)
    vo = np.stack([vo_grid] * n_t)
    return GriddedFieldSource(
        source_path="<synthetic>", time_name="time", time=TIME, lat_name="lat", lat=LAT,
        lon_name="lon", lon=LON, data={"uo": uo, "vo": vo}, fill_threshold=1e19,
    )

## 1. A small mock upstream particle payload

This is a stand-in for whatever the teammate's satellite-segmentation/particle-generation
module will eventually hand us -- plain records with the minimum fields this backtracking
module needs. `p3` is marked already invalid upstream, to show that case is preserved,
not silently dropped or reinterpreted.

In [16]:
print("required fields:", REQUIRED_UPSTREAM_FIELDS)

mock_upstream_payload = [
    {"particle_id": "p0", "latitude": 20.0, "longitude": 109.0, "observation_pixel_id": "px-0", "timestamp": T0},
    {"particle_id": "p1", "latitude": 20.0, "longitude": 110.0, "observation_pixel_id": "px-1", "timestamp": T0},
    {"particle_id": "p2", "latitude": 20.0, "longitude": 111.0, "observation_pixel_id": "px-1", "timestamp": T0},
    {"particle_id": "p3", "latitude": 20.0, "longitude": 112.0, "observation_pixel_id": "px-2", "timestamp": T0, "valid": False},
]
for r in mock_upstream_payload:
    print(r)

required fields: ('particle_id', 'latitude', 'longitude', 'observation_pixel_id', 'timestamp')
{'particle_id': 'p0', 'latitude': 20.0, 'longitude': 109.0, 'observation_pixel_id': 'px-0', 'timestamp': np.datetime64('2020-01-01T06:00:00.000000000')}
{'particle_id': 'p1', 'latitude': 20.0, 'longitude': 110.0, 'observation_pixel_id': 'px-1', 'timestamp': np.datetime64('2020-01-01T06:00:00.000000000')}
{'particle_id': 'p2', 'latitude': 20.0, 'longitude': 111.0, 'observation_pixel_id': 'px-1', 'timestamp': np.datetime64('2020-01-01T06:00:00.000000000')}
{'particle_id': 'p3', 'latitude': 20.0, 'longitude': 112.0, 'observation_pixel_id': 'px-2', 'timestamp': np.datetime64('2020-01-01T06:00:00.000000000'), 'valid': False}


## 2. Pass it through the adapter

In [17]:
adapted = adapt_upstream_particles(mock_upstream_payload)
print(f"{adapted.n_particles} particles adapted\n")
print(f"{'i':>2} {'particle_id':>11} {'pixel_id':>9} {'lat':>6} {'lon':>6} {'timestamp':>22} {'valid':>6}")
for i in range(adapted.n_particles):
    print(f"{i:>2} {str(adapted.particle_id[i]):>11} {str(adapted.observation_pixel_id[i]):>9} "
          f"{adapted.state.lat[i]:>6.2f} {adapted.state.lon[i]:>6.2f} {str(adapted.timestamp[i]):>22} "
          f"{str(bool(adapted.state.valid[i])):>6}")

4 particles adapted

 i particle_id  pixel_id    lat    lon              timestamp  valid
 0          p0      px-0  20.00 109.00 2020-01-01T06:00:00.000000000   True
 1          p1      px-1  20.00 110.00 2020-01-01T06:00:00.000000000   True
 2          p2      px-1  20.00 111.00 2020-01-01T06:00:00.000000000   True
 3          p3      px-2  20.00 112.00 2020-01-01T06:00:00.000000000  False


## 3. Verify required fields are actually enforced

A record missing any required field is rejected explicitly, not silently defaulted.

In [18]:
broken_payload = [dict(mock_upstream_payload[0])]
del broken_payload[0]["observation_pixel_id"]
try:
    adapt_upstream_particles(broken_payload)
    print("ERROR: should have been rejected")
except KeyError as exc:
    print(f"correctly rejected: {exc}")

correctly rejected: "upstream particle record 0 is missing required field(s): ['observation_pixel_id']"


## 4. Verify `observation_pixel_id` / `particle_id` preservation

The upstream identifiers must come through unchanged.

In [19]:
expected_particle_id = [r["particle_id"] for r in mock_upstream_payload]
expected_pixel_id = [r["observation_pixel_id"] for r in mock_upstream_payload]
print("particle_id preserved:", list(adapted.particle_id) == expected_particle_id)
print("observation_pixel_id preserved:", list(adapted.observation_pixel_id) == expected_pixel_id)
assert list(adapted.particle_id) == expected_particle_id
assert list(adapted.observation_pixel_id) == expected_pixel_id

particle_id preserved: True
observation_pixel_id preserved: True


## 5. Handoff into the existing T06-compatible representation

Take one adapted (still-valid) particle, run it through T06's unmodified
`run_backward_trajectory`, and confirm its `observation_pixel_id`/`particle_id` are still
correctly attached to the resulting trajectory.

In [20]:
ocean = make_ocean(np.full((3, 3), 0.3), np.full((3, 3), -0.1))
i = 1  # "p1", pixel "px-1", starts valid
traj = run_backward_trajectory(
    float(adapted.state.lat[i]), float(adapted.state.lon[i]), T0, n_steps=4, dt_seconds=DT,
    ocean_source=ocean, k_lat=LAT, k_lon=LON, k_field=ZERO_K, rng=np.random.default_rng(1),
    log_weight0=float(adapted.state.log_weight[i]),
)
tagged = tag_trajectory(traj, observation_pixel_id=adapted.observation_pixel_id[i], particle_id=adapted.particle_id[i])
print(f"particle_id={tagged.particle_id}, observation_pixel_id={tagged.observation_pixel_id}, "
      f"n_states={tagged.trajectory.n_states}, all valid={bool(np.all(tagged.trajectory.valid))}")
assert tagged.particle_id == "p1" and tagged.observation_pixel_id == "px-1"

particle_id=p1, observation_pixel_id=px-1, n_states=5, all valid=True


### The already-invalid upstream particle (`p3`) is preserved, not run through T06

T06's single-particle runner always starts from a fresh, valid state -- it has no "start
already invalid" mode, and T06 is not modified here to add one. The correct behavior is
that the adapter faithfully represents `p3` as already invalid (below), and a caller
simply does not invoke T06 for it -- there is nothing left to propagate.

In [21]:
j = 3  # "p3"
print(f"particle_id={adapted.particle_id[j]}, observation_pixel_id={adapted.observation_pixel_id[j]}, "
      f"valid={bool(adapted.state.valid[j])} (preserved from upstream, not dropped)")
assert not bool(adapted.state.valid[j])

particle_id=p3, observation_pixel_id=px-2, valid=False (preserved from upstream, not dropped)


## Summary / known limitations

- This module starts *after* upstream particle generation -- no segmentation, oil/clean
  classification, footprint generation, or particle initialization happens here.
- `adapt_upstream_particles` validates the minimum required fields (`particle_id`,
  `latitude`, `longitude`, `observation_pixel_id`, `timestamp`), rejects missing ones
  explicitly, and preserves an upstream `valid` flag when present.
- `particle_id` and `observation_pixel_id` survive unchanged through adaptation and
  through a full T06 backward trajectory.
- Neither T05 nor T06 was modified; this module only validates input and connects it to
  their existing, unchanged structures.
- The upstream schema has not been formally specified yet -- this adapter intentionally
  covers only the smallest clear contract, not a speculative full schema.